#  Feedback Intelligence system

In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\ADMIN\Documents\Guvi_Final_Project\RideFlow_Project/rideflow_datasets.csv")
df.head()

,ride_id,timestamp,pickup_zone,drop_zone,pickup_lat,pickup_long,drop_lat,drop_long,driver_id,customer_id,...,surge_multiplier,driver_rating,customer_rating,estimated_eta_min,actual_eta_min,ride_status,traffic_level,weather,driver_active,feedback_text
0,95.247911,2025-01-02 01:30:00,Anna Nagar,Adyar,12.880239,80.148410,13.028939,80.163941,1842.701958,6072.494896,...,1.001779,4.350624,4.037232,11.778023,18.304775,cancelled,low,clear,-0.036560,Driver was polite
1,439.187632,2025-01-05 12:45:00,T Nagar,Tambaram,13.092441,80.165458,13.142711,80.149376,1186.296422,5942.228896,...,1.193147,4.524196,3.324278,4.430894,13.343961,completed,low,cloudy,0.988999,Driver cancelled suddenly
2,876.685389,2025-01-09 23:00:00,Anna Nagar,Tambaram,12.817965,80.161839,12.943527,80.166040,1297.199801,5829.181415,...,2.008478,4.054085,4.979153,19.202891,12.039878,completed,low,rain,0.005750,Driver cancelled suddenly
3,275.337197,2025-01-03 19:30:00,T Nagar,Velachery,13.125103,80.143306,13.209127,80.126008,1765.474261,5429.619496,...,1.218528,3.689937,3.099466,18.711931,7.535792,completed,low,clear,1.023604,Good experience
4,106.743950,2025-01-02 02:30:00,Tambaram,Tambaram,13.143513,80.302596,13.078330,80.189672,1565.653849,5079.081677,...,1.497370,3.545512,3.073704,10.786351,12.104096,completed,high,cloudy,1.016716,Vehicle was not clean


In [2]:
# Keep only feedback_text
df = df[['feedback_text']].dropna()

# Remove empty feedback
df = df[df['feedback_text'].str.strip() != ""]

df.reset_index(drop=True, inplace=True)

# Create 3-class sentiment
def create_sentiment(text):
    text = str(text).lower()
    
    negative_words = ["rude", "cancel", "not clean", "late", "bad", "dirty"]
    positive_words = ["polite", "good", "nice", "friendly", "excellent"]
    
    if any(word in text for word in negative_words):
        return 0   # Negative
    elif any(word in text for word in positive_words):
        return 2   # Positive
    else:
        return 1   # Neutral

df['label'] = df['feedback_text'].apply(create_sentiment)
df['label'] = df['label'].astype(int)

df = df.sample(5000, random_state=42).reset_index(drop=True)

print("Total feedback rows after sampling:", len(df))
print(df['label'].value_counts())

Total feedback rows after sampling: 5000
label
0    2075
2    2002
1     923
Name: count, dtype: int64


In [3]:
print(df['label'].unique())
print(df['label'].value_counts())

[0 2 1]
label
0    2075
2    2002
1     923
Name: count, dtype: int64


In [4]:
df.head()

,feedback_text,label
0,Driver cancelled suddenly,0
1,Driver was polite,2
2,Driver cancelled suddenly,0
3,Ride was delayed,1
4,Driver was polite,2


In [5]:
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['feedback_text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Load tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=48
)

val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=48
)

C:\Users\ADMIN\anaconda3\envs\rideflow\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
import torch

class FeedbackDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
        
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item
    
    def __len__(self):
        return len(self.labels)

train_dataset = FeedbackDataset(train_encodings, train_labels.tolist())
val_dataset = FeedbackDataset(val_encodings, val_labels.tolist())

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

W0414 22:05:04.780000 15756 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [9]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

In [10]:
from tqdm import tqdm

epochs = 1  

for epoch in range(epochs):

    # --------------------
    # TRAINING
    # --------------------
    model.train()
    total_train_loss = 0

    train_bar = tqdm(train_loader, desc=f"Training Epoch {epoch+1}")

    for batch in train_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        batch.pop("token_type_ids", None)

        optimizer.zero_grad()

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        train_bar.set_postfix(train_loss=loss.item())

    avg_train_loss = total_train_loss / len(train_loader)

    # --------------------
    #  VALIDATION
    # --------------------
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation")

        for batch in val_bar:
            batch = {k: v.to(device) for k, v in batch.items()}
            batch.pop("token_type_ids", None)

            outputs = model(**batch)
            loss = outputs.loss

            total_val_loss += loss.item()
            val_bar.set_postfix(val_loss=loss.item())

    avg_val_loss = total_val_loss / len(val_loader)

    print("\n--------------------------------")
    print(f"Epoch {epoch+1}")
    print(f"Average Train Loss: {avg_train_loss:.4f}")
    print(f"Average Val Loss: {avg_val_loss:.4f}")
    print("--------------------------------")

Validation: 100%|██████████| 63/63 [00:58<00:00,  1.08it/s, val_loss=0.000665]


--------------------------------
Epoch 1
Average Train Loss: 0.0454
Average Val Loss: 0.0006
--------------------------------


In [11]:
model.save_pretrained("feedback_model")
tokenizer.save_pretrained("feedback_model")

('feedback_model\\tokenizer_config.json',
 'feedback_model\\special_tokens_map.json',
 'feedback_model\\vocab.txt',
 'feedback_model\\added_tokens.json',
 'feedback_model\\tokenizer.json')

In [12]:
from sklearn.metrics import accuracy_score, f1_score
import torch
import numpy as np

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        batch.pop("token_type_ids", None)

        outputs = model(**batch)
        logits = outputs.logits

        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))
print("F1 Score:", f1_score(all_labels, all_preds, average="weighted"))

Accuracy: 1.0
F1 Score: 1.0


In [13]:
def predict(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        pred = torch.argmax(logits, dim=1).item()

    labels = {0: "Negative", 1: "Neutral", 2: "Positive"}
    return labels[pred]

In [14]:
print(predict("Driver was very polite and helpful"))
print(predict("Driver cancelled the ride suddenly"))

Positive
Negative


In [ ]:
import json

label_map = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

with open("feedback_model/label_map.json", "w") as f:
    json.dump(label_map, f)